
# Seminario de Estadística II - Tarea 2 Parte A

**Integrantes del equipo:**
* Azpeitia Medina Samuel
* Castro Pérez Juan Antonio
* Rodríguez Rodríguez Donovan Zuriel 

### 1. Realize un análisis de la calidad de los datos de bronze a silver, elimine duplicados, conversión de tipos, missing vs ceros, lo que crea pertinente.

**Solución:**

In [0]:
# Cargamos nuestra tabla silver completa 
df_silver = spark.sql("SELECT * FROM dev.ciencias_data.silver_sessions")

# Cargamos la tabla bronce 
df_bronze = spark.sql("SELECT * FROM dev.ciencias_data.bronze_sessions")

# Visualizamos la tabla Silver
print("Mostrando primeros 20 registros de la tabla Silver:")
df_silver.limit(20).display()

# Visualizamos la tabla Bronce
print("Mostrando primeros 20 registros de la tabla Bronce:")
df_bronze.limit(20).display()

### 2. Construya un FeatureStore para los datos de sesiones usando FeatureEngineeringClient() en databricks.

* **Construya las variables que crea pertinentes como duración de la sesión, bytes trasmitidos por unidad de tiempo, el tamaño promedio de los paquetes, ratio de bytes recibidos entre enviados etc.**

**Solución:**

Crear variables derivadas (Feature Engineering)

Duración de sesión

In [0]:
silver_df = silver_df.withColumn(
    "session_duration",
    unix_timestamp("end_time") - unix_timestamp("start_time")
)

* **Registre el FeatureStore en Unity Catalog como se vio en clase.**

**Solución:**

In [0]:
#Código

### 3. Cree un Pipeline de datos donde implemente las diferentes técnicas de Feature Engineering vistos en la clase, seguún corresponda la naturaleza de la variable.

* **Construya el “set de entrenamiento” usando FeatureLookop y FeatureEngineeringClient.**

**Solución:**

In [0]:
# Código

* **Realize un análisis de conglomerados, usando las diferentes técnicas como K-Means y DBSCAN.**

**Solución:**

In [0]:
# Código

* **Implemente un modelo de detecci ́on de anomalías usando Isolation Forest.**

**Solución:**

In [0]:
# Código

* **Comparta sus conclusiones de manera detallada, incluyendo como determino los perfiles de comportamiento y grupos.**

**Solución:**

In [0]:
# Código

* **Valide la calidad de su clusterización, ¿Qué observa?**

**Solución:**

### 4. Para el ejemplo visto en clase sobre clustering jerárquico, realize el mismo ejemplo utilizando el enlace completo.

**Solución:**

## Introducción

El análisis de conglomerados o **clustering** es una técnica de aprendizaje no supervisado cuyo objetivo es agrupar observaciones de forma que los elementos dentro de un mismo grupo sean más similares entre sí que con los elementos de otros grupos.

Uno de los enfoques más utilizados es el **clustering jerárquico**, el cual construye una jerarquía de agrupamientos mediante un proceso iterativo de fusión de clusters. Inicialmente cada observación se considera como un cluster independiente y posteriormente se combinan los clusters más cercanos hasta formar un único grupo.

En este ejercicio se aplica el método de **clustering jerárquico utilizando enlace completo (Complete Linkage)** para agrupar delegaciones de la Ciudad de México con el objetivo de determinar cómo podrían organizarse en oficinas administrativas para la expedición de licencias de conducir.

---

## Datos del problema

Se cuenta con una matriz de distancias entre seis delegaciones:

- Milpa Alta  
- Tláhuac  
- Iztapalapa  
- Tlalpan  
- Xochimilco  
- Coyoacán  

La matriz de distancias (en kilómetros) es la siguiente:

|               | Milpa Alta | Tláhuac | Iztapalapa | Tlalpan | Xochimilco | Coyoacán |
|---------------|------------|---------|------------|---------|------------|----------|
| Milpa Alta    | 0          | 33.2    | 31.3       | 25.7    | 11.4       | 39.6     |
| Tláhuac       | 33.2       | 0       | 8.6        | 18.5    | 15.8       | 15.4     |
| Iztapalapa    | 31.3       | 8.6     | 0          | 18.7    | 16.0       | 15.3     |
| Tlalpan       | 25.7       | 18.5    | 18.7       | 0       | 9.3        | 11.1     |
| Xochimilco    | 11.4       | 15.8    | 16.0       | 9.3     | 0          | 15.3     |
| Coyoacán      | 39.6       | 15.4    | 15.3       | 11.1    | 15.3       | 0        |

Estas distancias representan qué tan separadas se encuentran las delegaciones entre sí.

---

## Método de enlace completo

En el **método de enlace completo (Complete Linkage)** la distancia entre dos clusters se define como la **mayor distancia entre cualquier par de elementos pertenecientes a ambos clusters**.

Matemáticamente:

\[
d(A,B) = \max_{x_i \in A, \, x_j \in B} d(x_i, x_j)
\]

Este método tiende a generar clusters **más compactos**, ya que considera el peor caso de separación entre los elementos de dos grupos.

---

## Paso 1: Inicialización de clusters

Inicialmente cada delegación se considera un cluster independiente:

\[
\{\text{Milpa Alta}\},\; \{\text{Tláhuac}\},\; \{\text{Iztapalapa}\},\; \{\text{Tlalpan}\},\; \{\text{Xochimilco}\},\; \{\text{Coyoacán}\}
\]

---

## Paso 2: Primera fusión de clusters

Se busca la **distancia mínima en la matriz**.

La menor distancia observada es:

\[
d(\text{Tláhuac}, \text{Iztapalapa}) = 8.6
\]

Por lo tanto se fusionan estas dos delegaciones:

\[
\{\text{Tláhuac, Iztapalapa}\}
\]

Los clusters ahora son:

\[
\{\text{Tláhuac, Iztapalapa}\},\; \{\text{Milpa Alta}\},\; \{\text{Tlalpan}\},\; \{\text{Xochimilco}\},\; \{\text{Coyoacán}\}
\]

---

## Paso 3: Recalcular distancias usando enlace completo

Se deben recalcular las distancias entre el nuevo cluster y los demás.

Por ejemplo, para calcular la distancia entre:

\[
\{\text{Tláhuac, Iztapalapa}\} \quad\text{y}\quad \{\text{Xochimilco}\}
\]

se usa la distancia máxima:

\[
\begin{aligned}
d(\{\text{Tláhuac, Iztapalapa}\}, \text{Xochimilco}) &= \max\big( d(\text{Tláhuac},\text{Xochimilco}),\; d(\text{Iztapalapa},\text{Xochimilco}) \big) \\
&= \max(15.8,\; 16.0) = 16.0
\end{aligned}
\]

Este procedimiento se repite para todos los clusters restantes.

---

## Paso 4: Segunda fusión

La siguiente menor distancia en la matriz es:

\[
d(\text{Tlalpan}, \text{Xochimilco}) = 9.3
\]

Por lo tanto se forma el cluster:

\[
\{\text{Tlalpan, Xochimilco}\}
\]

Los clusters quedan:

\[
\{\text{Tláhuac, Iztapalapa}\},\; \{\text{Tlalpan, Xochimilco}\},\; \{\text{Milpa Alta}\},\; \{\text{Coyoacán}\}
\]

---

## Paso 5: Recalcular distancias

Se vuelve a calcular la distancia entre clusters.

Ejemplo:

\[
\begin{aligned}
d(\{\text{Tlalpan, Xochimilco}\}, \text{Coyoacán}) &= \max\big( d(\text{Tlalpan},\text{Coyoacán}),\; d(\text{Xochimilco},\text{Coyoacán}) \big) \\
&= \max(11.1,\; 15.3) = 15.3
\end{aligned}
\]

---

## Paso 6: Tercera fusión

La menor distancia restante corresponde a:

\[
d(\text{Milpa Alta}, \text{Xochimilco}) = 11.4
\]

Por lo tanto Milpa Alta se integra al cluster:

\[
\{\text{Milpa Alta, Tlalpan, Xochimilco}\}
\]

Los clusters ahora son:

\[
\{\text{Tláhuac, Iztapalapa}\},\; \{\text{Milpa Alta, Tlalpan, Xochimilco}\},\; \{\text{Coyoacán}\}
\]

---

## Paso 7: Cuarta fusión

La delegación **Coyoacán** se une al cluster más cercano, generando:

\[
\{\text{Milpa Alta, Tlalpan, Xochimilco, Coyoacán}\}
\]

Finalmente quedan dos clusters principales:

\[
\{\text{Tláhuac, Iztapalapa}\}
\]
\[
\{\text{Milpa Alta, Tlalpan, Xochimilco, Coyoacán}\}
\]

---

## Resultado final

El algoritmo de clustering jerárquico con enlace completo produce dos conglomerados principales.

### Cluster 1
- Tláhuac  
- Iztapalapa  

### Cluster 2
- Milpa Alta  
- Tlalpan  
- Xochimilco  
- Coyoacán  

---

## Interpretación de los resultados

Los resultados del análisis muestran que las delegaciones pueden dividirse en dos grupos geográficamente coherentes.

El primer cluster agrupa a **Tláhuac e Iztapalapa**, delegaciones ubicadas en la zona oriente de la ciudad y que presentan menor distancia entre sí.

El segundo cluster agrupa a **Milpa Alta, Tlalpan, Xochimilco y Coyoacán**, las cuales presentan mayor cercanía relativa entre ellas.

Esta agrupación permitiría al gobierno establecer **dos oficinas administrativas para la expedición de licencias**, optimizando la distribución de los ciudadanos y reduciendo las distancias promedio de traslado.

In [0]:
# Código